# Polar images and grids

A digital image is a set of samples of a continuous function, taken at the points of a sampling grid. This notebook introduces sampling grids in general. It then moves an ordinary image from a Cartesian grid to a polar grid and back, and fixes the angle convention every later notebook relies on.

In [ ]:
# Draw every figure below the cell that makes it
%matplotlib inline

# NumPy for arrays, matplotlib for figures, and PyPFT itself
import matplotlib.pyplot as plt
import numpy as np

import pypft

## Sampling grids

A sampling grid is the set of points at which a continuous function $f$ is measured. The array of those measurements is all a computer ever holds of $f$.

A **Cartesian grid** places its points $(x_i, y_j)$ on a lattice of rows and columns, with equal spacing along $x$ and along $y$. Every pixel of an ordinary image covers the same square area.

A **polar grid** places its points $(r_k, \theta_m)$ on concentric rings of constant radius $r$ and on spokes of constant angle $\theta$. A function written in polar coordinates, $f(r, \theta)$, is naturally sampled on such a grid.

The polar Fourier transform treats the two polar directions separately. It takes a DFT over the angular samples, and then a Hankel transform over the radial samples along each angle, one per angular harmonic. Both steps need the samples arranged by angle, and by radius along each angle. The points of a Cartesian lattice are arranged by row and column instead, so the transform needs its samples on a polar grid.

A grid is **uniform** when consecutive samples are equally spaced, that is, when $\Delta r$ and $\Delta\theta$ are constant. A uniform polar grid is the simplest polar grid. The transform's own grid is uniform in angle but not in radius. It has the same equally spaced spokes as a uniform grid with as many angles, but its radial sample positions come from the zeros of Bessel functions, and they differ from one spoke to the next.

Three grids therefore play a role in this package, each with its own job:

- **The uniform polar grid** is what your data is on. An image resampled to polar coordinates, or a polar acquisition, has the same equally spaced radii on every spoke, so every radial index is a true ring. `pypft.cartesian_to_polar` and `pypft.polar_to_cartesian`, used below, work on this grid.
- **The transform's grid**, `pypft.PolarGrid`, is where the transform needs its samples if its output is meant to approximate the continuous Fourier transform. The last section of this notebook previews it, and [02_sampling_grids](02_sampling_grids.ipynb) introduces it properly.
- **The bridge** between them is interpolation along each spoke, from the uniform radii onto the transform's radii. Since both grids share their spokes, nothing has to be interpolated across angles. [02_sampling_grids](02_sampling_grids.ipynb) measures that this gives the same transform as sampling on the transform's grid directly.

The figure shows a Cartesian lattice and a uniform polar grid covering the same disk:

In [ ]:
# A Cartesian lattice: 13 x 13 equally spaced points on the square [-1, 1] x [-1, 1]
lattice = np.linspace(start=-1.0, stop=1.0, num=13)
lattice_x, lattice_y = np.meshgrid(lattice, lattice)

# A uniform polar grid: 6 equally spaced radii x 24 equally spaced angles
radii = np.linspace(start=1.0 / 6, stop=1.0, num=6)
angles = np.arange(24) * (2 * np.pi / 24)
polar_r, polar_theta = np.meshgrid(radii, angles)

# Both grids in the same Cartesian axes, with the unit circle for reference
circle = np.linspace(start=0.0, stop=2 * np.pi, num=200)
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(8, 4))
axes[0].scatter(x=lattice_x, y=lattice_y, s=8)
axes[0].set_title(label="Cartesian grid")
axes[0].set_xlabel(xlabel="x")
axes[0].set_ylabel(ylabel="y")
axes[1].scatter(x=polar_r * np.cos(polar_theta), y=polar_r * np.sin(polar_theta), s=8)
axes[1].set_title(label="uniform polar grid")
axes[1].set_xlabel(xlabel="x")
axes[1].set_ylabel(ylabel="y")
for ax in axes:
    ax.plot(np.cos(circle), np.sin(circle), color="gray", linewidth=0.8)
    ax.set_aspect(aspect="equal")
fig.tight_layout()
plt.show()

## A synthetic test image

Angular wedges and concentric rings make both polar directions easy to see.

PyPFT measures angles directly on image coordinates. For a pixel at column $c$ and row $\ell$ of an image centered at $(c_0, \ell_0)$, the offsets are $x = c - c_0$, which grows to the right, and $y = \ell - \ell_0$, which grows downward like the row index. The pixel's polar coordinates are

$$r = \sqrt{x^2 + y^2}, \qquad \theta = \operatorname{atan2}(y, x).$$

The angle therefore increases from the $+x$ direction (right) toward the $+y$ direction (down): $\theta = \pi/2$ points straight down on screen, and the positive direction is clockwise as the image is drawn. `cartesian_to_polar`, `polar_to_cartesian`, and `sample_cartesian` all use this convention.

In [ ]:
# Image size in pixels, and its center
size = 256
center = size / 2

# Row and column index of every pixel
rows, cols = np.mgrid[0:size, 0:size]

# Offsets from the center: x grows to the right, y grows downward
x = cols - center
y = rows - center

# Polar coordinates of every pixel, in PyPFT's angle convention
radius = np.hypot(x, y)
angle = np.arctan2(y, x)

# Eight wedges (bright where sin(4 angle) > 0) and rings (bright where sin(r / 6) > 0)
wedges = (np.sin(4 * angle) > 0).astype(float)
rings = (np.sin(radius / 6) > 0).astype(float)

# Average the two patterns and blank everything near and beyond the image border
image = 0.5 * wedges + 0.5 * rings
image[radius > 0.95 * size / 2] = 0.0

# Show the image
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(X=image, cmap="gray")
ax.set_title(label="Cartesian test image")
plt.show()

## To polar and back

`cartesian_to_polar(image, n_radial, n_angular)` returns an array of shape `(n_radial, n_angular)`, PyPFT's `(radial, angular)` layout (`pypft.PolarAxis`). Row $k$ is a radius and column $m$ is an angle. The radii are equally spaced from the center out to the largest circle that fits in the image.

The angular axis is **centered**. With $N_\theta$ angular samples, column $m$ holds the angle

$$\theta_m = \left(m - \lfloor N_\theta / 2 \rfloor\right) \frac{2\pi}{N_\theta},$$

so angle $0$ sits in the middle column, negative angles to its left, and positive angles to its right.

`polar_to_cartesian(polar, height, width)` is the inverse, given the original image's height and width. It can only fill pixels inside the largest circle that fits in the image, since the polar grid holds no samples beyond it, so the corners outside that circle are set to zero here.

In [ ]:
# A uniform polar grid of 128 radii x 96 angles
n_radial, n_angular = 128, 96

# Cartesian -> polar -> Cartesian
polar = pypft.cartesian_to_polar(image=image, n_radial=n_radial, n_angular=n_angular)
reconstructed = pypft.polar_to_cartesian(polar=polar, height=size, width=size)

# Original, polar array (radius down, angle across), and round trip
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(12, 4))
axes[0].imshow(X=image, cmap="gray")
axes[0].set_title(label="original")
axes[1].imshow(X=polar, cmap="gray", aspect="auto")
axes[1].set_title(label="polar (radius down, angle across)")
axes[1].set_xlabel(xlabel="angular index m")
axes[1].set_ylabel(ylabel="radial index k")
axes[2].imshow(X=reconstructed, cmap="gray")
axes[2].set_title(label="reconstructed")
fig.tight_layout()
plt.show()

The round trip interpolates twice, once onto the polar grid and once back, so it is close to the original but not exact. The error map shows where the two differ. Almost all of the error sits on the straight edges between wedges, and it widens with the radius. Neighboring angular samples on a ring of radius $r$ are $r\,\Delta\theta$ apart, so the farther from the center, the coarser the polar grid is along the ring and the more an edge across the ring is blurred. The ring edges run along the rings, and the radial samples are close together, so they barely show.

In [ ]:
# Absolute round-trip error at every pixel
error = np.abs(image - reconstructed)

# Mean error over the pixels inside the inscribed circle
disk = radius <= size / 2 - 1
mean_error = float(error[disk].mean())

# Show the error map
fig, ax = plt.subplots(figsize=(5, 4))
shown = ax.imshow(X=error, cmap="magma")
ax.set_title(label=f"round-trip error, mean {mean_error:.3f}")
fig.colorbar(mappable=shown, ax=ax)
plt.show()

## The angular origin

Combining the angle convention with the centered angular axis predicts exactly where an angle $\theta_0$ lands: in column

$$m_0 = \left(\lfloor N_\theta / 2 \rfloor + \operatorname{round}\!\left(\frac{\theta_0 N_\theta}{2\pi}\right)\right) \bmod N_\theta.$$

A single thin wedge at $\theta_0 = \pi/2$, pointing straight down on screen, tests this. Summing its polar image over the radius gives an angular profile that peaks in the column the wedge landed in.

In [ ]:
# A 10-degree-wide wedge centered on theta_0 = pi / 2 (straight down on screen)
theta_0 = np.pi / 2
angle_offset = (angle - theta_0 + np.pi) % (2 * np.pi) - np.pi  # wrapped to [-pi, pi)
wedge = (np.abs(angle_offset) < np.deg2rad(5)) & (radius < 0.9 * size / 2)

# The wedge on the uniform polar grid
wedge_polar = pypft.cartesian_to_polar(
    image=wedge.astype(float), n_radial=n_radial, n_angular=n_angular
)

# Angular profile: the polar image summed over the radius
profile = wedge_polar.sum(axis=0)

# The column the wedge landed in, and the column the formula predicts
measured_index = int(np.argmax(profile))
predicted_index = (n_angular // 2 + round(theta_0 * n_angular / (2 * np.pi))) % n_angular

# The wedge, its polar image, and its angular profile, with both columns marked
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(13, 4))
axes[0].imshow(X=wedge, cmap="gray")
axes[0].set_title(label=r"wedge at $\theta_0 = \pi / 2$")
axes[1].imshow(X=wedge_polar, cmap="gray", aspect="auto")
axes[1].set_title(label="polar image")
axes[1].set_xlabel(xlabel="angular index m")
axes[1].set_ylabel(ylabel="radial index k")
axes[2].plot(profile)
axes[2].set_title(label="angular profile")
axes[2].set_xlabel(xlabel="angular index m")
axes[2].set_ylabel(ylabel="intensity summed over the radius")
for ax in axes[1:]:
    ax.axvline(x=measured_index, color="tab:orange", linewidth=3, label="measured")
    ax.axvline(x=predicted_index, color="tab:cyan", linestyle="--", label="predicted")
axes[2].legend()
fig.tight_layout()
plt.show()

The measured and predicted columns coincide. As a final check:

In [ ]:
# The measured column must equal the predicted one exactly
assert measured_index == predicted_index
measured_index, predicted_index

## A preview of the transform's own grid

`pypft.PolarGrid` is the grid the transform itself samples on, and `pypft.sample_cartesian` evaluates an image at its points. The result has one row per angle, shape `(n_angular, n_radial)`, so it is transposed to the `(radial, angular)` layout. Wrapping the samples in a `pypft.PolarSpatialAngularSignal` gives access to PyPFT's plotting functions. `pypft.plot_signal` draws the magnitude and phase over the polar sample indices, and `pypft.render_cartesian` draws the samples back onto a pixel grid for display. The phase of a real, non-negative image is zero everywhere.

In the index plot the rings are no longer horizontal. The transform's grid has the same spokes as a uniform grid with as many angles, but each spoke places its radial samples differently, so the same radial index sits at a different radius on different spokes. A row of the index table is therefore not a ring, unlike a row of `cartesian_to_polar`'s output. To transform a uniform polar array such as `cartesian_to_polar`'s, interpolate it along each spoke onto the transform's grid with `pypft.resample_uniform_polar`. [02_sampling_grids](02_sampling_grids.ipynb) explains how this grid places its samples.

In [ ]:
# The transform's own grid, reaching out to the rim of the test image's disk
grid = pypft.PolarGrid(n_radial=192, n_angular=96, R=0.95 * size / 2)

# Sample the test image on it and transpose to (radial, angular)
spatial = pypft.PolarSpatialAngularSignal(
    values=pypft.sample_cartesian(image=image, grid=grid).T, grid=grid
)

# Magnitude and phase over the polar sample indices
pypft.plot_signal(signal=spatial)
plt.show()

# The samples drawn back as a Cartesian image
fig, ax = plt.subplots(figsize=(4, 4))
pypft.render_cartesian(signal=spatial, height=size, width=size, ax=ax)
plt.show()

The overlay below makes the difference visible on a small grid. Both grids have 12 spokes at the same angles and 8 samples per spoke. The uniform grid (gray) puts its samples at the same radii on every spoke. The transform's grid (blue) moves them along the spoke, differently on every spoke, most of all near the center. Three spokes are highlighted, and the dashed line joins the third radial sample of every spoke of the transform's grid: it would be a circle on the uniform grid, but here it is not.

In [ ]:
# Two grids with the same 12 spokes and 8 samples per spoke, out to R = 1
small_grid = pypft.PolarGrid(n_radial=8, n_angular=12, R=1.0)
uniform_radii = np.arange(start=1, stop=9) / 9

# Every sample point of both grids, in image coordinates (y grows downward)
spokes = small_grid.theta[:, np.newaxis]
uniform_x, uniform_y = uniform_radii * np.cos(spokes), uniform_radii * np.sin(spokes)
grid_x, grid_y = small_grid.r * np.cos(spokes), small_grid.r * np.sin(spokes)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(x=uniform_x, y=uniform_y, s=30, color="gray", label="uniform polar grid")
ax.scatter(x=grid_x, y=grid_y, s=12, color="tab:blue", label="PolarGrid")

# Highlight three spokes of the transform's grid with their own colors
for row, color in ((0, "tab:red"), (3, "tab:orange"), (6, "tab:green")):
    ax.plot([0, np.cos(small_grid.theta[row])], [0, np.sin(small_grid.theta[row])],
            color=color, linewidth=0.8)
    ax.scatter(x=grid_x[row], y=grid_y[row], s=40, color=color,
               label=f"spoke at {np.rad2deg(small_grid.theta[row]):.0f} degrees")

# Join radial index 2 across every spoke of the transform's grid: not a circle
k = 2
ax.plot(np.append(grid_x[:, k], grid_x[0, k]), np.append(grid_y[:, k], grid_y[0, k]),
        color="tab:blue", linestyle="--", label=f"radial index {k} of PolarGrid")

# Image convention: y grows downward
ax.set_xlim(left=-1.1, right=1.1)
ax.set_ylim(bottom=1.1, top=-1.1)
ax.set_aspect(aspect="equal")
ax.set_xlabel(xlabel="x")
ax.set_ylabel(ylabel="y (down)")
ax.set_title(label="same spokes, different radii")
ax.legend(loc="upper left", fontsize="small")
plt.show()

## Where to go next

[02_sampling_grids](02_sampling_grids.ipynb) takes `PolarGrid` apart: why its radial samples are non-uniform and differ from one spoke to the next, what that means for uniformly sampled polar data, why it leaves a gap at the center, and how to check whether a grid is large enough.